# Week 1 · Day 2 — Lab 1
## The ndarray: Shape, dtype & Memory

Everything in the AI stack — embeddings, model weights, token batches, eval
scores — is an **ndarray** underneath. Before you can vectorize anything, you
have to *see* an array the way NumPy does: a typed, contiguous block of memory
described by a little bundle of metadata (shape, dtype, strides). This lab makes
that metadata concrete and hands-on.

### Learning objectives
By the end of this lab you can:
1. Inspect an array's `shape`, `ndim`, `size`, `dtype`, `strides`, and `nbytes`, and say what each one means.
2. Reshape, `ravel`, and `flatten` arrays — and **predict** when you get a *view* vs. a *copy*.
3. Choose and convert dtypes deliberately (`float64` ↔ `float32`), and avoid the aliases NumPy 2.0 removed.
4. Explain C-order vs. F-order contiguity and why it can cause silent copies.
5. Persist and reload arrays with `np.save`, `np.load`, and `np.savez_compressed`.

### Time budget — ~70 min
| Segment | Time |
|---|---|
| Framing & objectives | 5 min |
| **A.** Load & inspect (guided) | 12 min |
| **B.** Reshape · views vs. copies | 16 min |
| **C.** dtype & memory (NEP 50) | 16 min |
| **D.** Contiguity: C vs. F | 10 min |
| **E.** Save & load | 8 min |
| Wrap-up + stretch | 3 min |

### Files you need (in a `data/` folder next to this notebook)
- `lab1_image_batch.npy` — a batch of 64 synthetic 28×28 grayscale glyphs (`uint8`).
- `lab1_layer_weights.npy` — a synthetic dense-layer weight matrix, shape (256, 128), `float64`.

Run the setup cell, then work top to bottom. Each **Exercise** has a `check(...)`
cell right after it; finish the exercise until the check prints **PASS**.


In [ ]:
%%python data/generate_lab_data.py

from pathlib import Path
DATA = Path("data")

In [ ]:
%pip install --upgrade numpy

import numpy as np
print("NumPy", np.__version__)   # target curriculum: NumPy 2.x on Python 3.13

def check(label, predicate):
    """Soft check: prints a result and never raises.

    Using a zero-argument lambda lets us defer evaluation, so an unfinished
    exercise prints a clean failure instead of crashing the whole notebook.
    """
    try:
        ok = bool(predicate())
    except Exception as exc:
        ok = False
        label = f"{label}  (raised {type(exc).__name__}: {exc})"
    print(("PASS " if ok else "FAIL "), label)
    return ok

## Part A — Load and inspect an array  *(guided / we-do)*

The single most useful habit in NumPy: when a new array lands on your desk,
**print its metadata first**. Let's load the image batch and read it the way
NumPy stores it.


In [ ]:
images = np.load(DATA / "lab1_image_batch.npy")

print("shape :", images.shape)    # (batch, height, width)
print("ndim  :", images.ndim)     # number of axes
print("size  :", images.size)     # total number of elements
print("dtype :", images.dtype)    # element type
print("strides:", images.strides) # bytes to step along each axis
print("nbytes:", images.nbytes)   # total bytes of the data buffer

**Read that output like an engineer:**
- `shape (64, 28, 28)` → 64 images, each 28×28.
- `dtype uint8` → 1 byte per element, values 0–255 (perfect for pixels).
- `strides` → how many *bytes* NumPy jumps to move one step along each axis.
  For a contiguous `uint8` array the last axis steps 1 byte, the middle axis
  steps 28 bytes (one row), the first axis steps 28×28 = 784 bytes (one image).
- `nbytes = size × itemsize` → the real memory cost.


### Exercise A1 — Predict, then verify `nbytes`
Without calling `.nbytes`, compute the expected memory footprint of `images`
from `size` and the per-element size, then confirm it matches.

Fill in the two variables below.


In [ ]:
!python3 -c "import numpy as np; a=np.load('data/lab1_image_batch.npy'); print(a.shape, a.dtype); print(a)"

In [ ]:
# TODO: bytes used by ONE element of `images` (hint: images.dtype.itemsize)
bytes_per_element = None

# TODO: total bytes = number of elements * bytes per element
total_bytes = None

In [ ]:
check("A1: bytes_per_element is 1 (uint8)", lambda: bytes_per_element == 1)
check("A1: total_bytes matches images.nbytes", lambda: total_bytes == images.nbytes)

## Part B — Reshape, ravel, flatten — *view or copy?*

Reshaping usually just rewrites the **strides** metadata and points at the same
buffer (a **view**, O(1), no data moved). But some operations are forced to
**copy**. Knowing which is which prevents a whole class of silent bugs.

`np.shares_memory(a, b)` is the definitive test.


In [ ]:
a = np.arange(12)
b = a.reshape(3, 4)
print("reshape view? ", np.shares_memory(a, b))   # True — no copy
b[0, 0] = 999
print("a[0] after editing the reshape:", a[0])     # 999 — same buffer!

### Exercise B1 — Flatten a batch into feature vectors
Many models want each 28×28 image as a flat 784-length vector, i.e. shape
`(64, 784)`. Reshape `images` into `flat_images` of that shape, and confirm it
is a **view** of `images` (no copy).

> Use `-1` to let NumPy infer the second dimension if you like.


In [ ]:
# TODO: reshape `images` (64, 28, 28) -> (64, 784)
flat_images = None

In [ ]:
check("B1: flat_images shape is (64, 784)",
      lambda: flat_images.shape == (64, 784))
check("B1: flat_images is a VIEW of images",
      lambda: np.shares_memory(flat_images, images))

### Exercise B2 — `flatten()` vs `ravel()`
`ravel()` returns a *view* when it can; `flatten()` **always** returns a fresh
copy. Demonstrate the difference: create `raveled` and `flattened` from
`images`, then set `raveled_is_view` and `flattened_is_view` to the booleans
that `np.shares_memory` reports.


In [ ]:
# TODO: ravel and flatten `images`, then test memory sharing for each
raveled = None
flattened = None
raveled_is_view = None
flattened_is_view = None

In [ ]:
check("B2: ravel() returned a view", lambda: raveled_is_view is True)
check("B2: flatten() returned a copy", lambda: flattened_is_view is False)

## Part C — dtype & memory: choose it on purpose

Two NumPy-2.x facts the cohort must internalize:

1. **NEP 50 type promotion.** A Python scalar no longer widens an array's dtype.
   `int8_array + 1` *stays* `int8` (the scalar is "weakly typed"). Older
   tutorials assumed the opposite. **Be explicit with dtypes.**
2. **Removed aliases.** `np.int`, `np.float`, `np.bool`, `np.complex` were
   **removed** in NumPy 2.0 — they raise `AttributeError`. Use `np.int64`,
   `np.float64`, `np.bool_`, `np.complex128`, or plain Python `int`/`float`.


In [ ]:
x = np.array([1, 2, 3], dtype=np.int8)
print("(int8 array + 1).dtype =", (x + 1).dtype)   # int8 under NEP 50

# The removed alias raises — proving it's gone (we catch it on purpose):
try:
    np.zeros(3, dtype=np.float)     # type: ignore[attr-defined]
except AttributeError as e:
    print("np.float is gone:", str(e).splitlines()[0])

### Exercise C1 — Halve the memory of a weight matrix
Model weights are often stored as `float32` to halve memory vs. `float64`.
Load `lab1_layer_weights.npy` (float64), cast it to `float32`, and verify the
byte count is exactly half.


In [ ]:
weights64 = np.load(DATA / "lab1_layer_weights.npy")

# TODO: cast to float32 with .astype(...)
weights32 = None

# TODO: record the two byte counts
bytes64 = None
bytes32 = None

In [ ]:
check("C1: weights32 is float32", lambda: weights32.dtype == np.float32)
check("C1: float32 uses exactly half the bytes", lambda: bytes32 * 2 == bytes64)

### Exercise C2 — `casting="safe"` catches precision loss early
In a data pipeline you want *accidental* precision loss to fail loudly. The
`casting="safe"` argument to `astype` raises a `TypeError` when the cast would
lose information. Try to safely cast the float weights to `int8` — capture the
fact that it raised by setting `safe_cast_failed = True` in the `except`.


In [ ]:
safe_cast_failed = None
# TODO: wrap weights64.astype(np.int8, casting="safe") in try/except TypeError,
#       and set safe_cast_failed = True inside the except block.

In [ ]:
check("C2: safe cast float64->int8 was refused", lambda: safe_cast_failed is True)

## Part D — Memory layout: C-order vs. F-order

NumPy defaults to **C-order** (row-major): the last axis is contiguous in
memory. **F-order** (column-major) makes the first axis contiguous. Most NumPy
ops and PyTorch tensors are C-order; some BLAS/LAPACK routines prefer F-order,
and a mismatch can trigger a silent copy. `array.flags` tells you the truth.


In [ ]:
m = np.arange(6).reshape(2, 3)
print("C_CONTIGUOUS:", m.flags["C_CONTIGUOUS"], " strides:", m.strides)

mf = np.asfortranarray(m)
print("F_CONTIGUOUS:", mf.flags["F_CONTIGUOUS"], " strides:", mf.strides)

### Exercise D1 — Transpose flips contiguity
Transposing a C-contiguous 2-D array (just a stride swap — a **view**) yields an
array that is *F*-contiguous, not C-contiguous. Take `flat_images` from B1,
transpose it into `flat_T`, and record its contiguity flags.


In [ ]:
# TODO: transpose flat_images (use .T)
flat_T = None
flat_T_is_C = None     # TODO: flat_T.flags["C_CONTIGUOUS"]
flat_T_is_F = None     # TODO: flat_T.flags["F_CONTIGUOUS"]
flat_T_shares = None   # TODO: np.shares_memory(flat_T, flat_images)

In [ ]:
check("D1: transpose is a VIEW (no copy)", lambda: flat_T_shares is True)
check("D1: transposed array is F-contiguous, not C", 
      lambda: (flat_T_is_F is True) and (flat_T_is_C is False))

## Part E — Save and load

`.npy` stores one array (binary, fast). `.npz` stores several under names.
`savez_compressed` trades CPU for disk and is the right default for large
arrays you'll ship to teammates.


### Exercise E1 — Bundle and round-trip
Save `images` and `weights32` together into a compressed archive
`my_bundle.npz` (keys `images` and `weights`), reload it, and verify both
arrays survive the round trip unchanged.


In [ ]:
out_path = "my_bundle.npz"
# TODO: np.savez_compressed(out_path, images=..., weights=...)

# TODO: reload with np.load(out_path) into `archive`
archive = None

# TODO: pull arrays back out
images_back = None
weights_back = None

In [ ]:
check("E1: images survived the round trip",
      lambda: np.array_equal(images_back, images))
check("E1: weights survived the round trip",
      lambda: np.array_equal(weights_back, weights32) and weights_back.dtype == np.float32)

## Stretch goals *(for fast finishers)*

**S1 — `-1` inference.** Reshape `images` to `(28, -1)` and explain in a comment
what the inferred dimension is and why.

**S2 — Make the copy explicit.** `flat_images` is a view, so editing it edits
`images`. Produce an *independent* flattened copy and prove with
`np.shares_memory` that editing it leaves `images` untouched.


In [ ]:
# S1 + S2 (optional)
stretch_reshape = None       # TODO: images.reshape(28, -1); what is the -1?
independent_flat = None      # TODO: a copy that does NOT share memory with images

In [ ]:
check("S1: reshape (28, -1) -> (28, 1792)",
      lambda: stretch_reshape.shape == (28, 1792))
check("S2: independent_flat does not share memory with images",
      lambda: not np.shares_memory(independent_flat, images))

## Wrap-up — what you can now do

- Read an array's `shape / ndim / size / dtype / strides / nbytes` and explain each.
- Predict **view vs. copy** for reshape, ravel, flatten, transpose — and confirm with `np.shares_memory`.
- Convert dtypes deliberately and know that **NEP 50** keeps scalars from widening your arrays.
- Spot C- vs F-contiguity and why it matters for performance.
- Persist arrays with `np.save` / `np.savez_compressed` and round-trip them safely.

**Next:** Lab 2 turns these typed blocks of memory into *fast* computation —
vectorization and broadcasting, no Python loops.
